# Exercice 6 ★★★★★ — Une vraie relation entre deux tables

**Comment faire :**
1. Lancez la cellule des imports.
2. Écrivez votre code dans la cellule « Votre code », sous le `TODO`, puis lancez-la.
3. Lancez la cellule « Vérification » et lisez les messages ❌ : ils disent ce qui manque.
4. Corrigez et relancez autant de fois que nécessaire.

Bloqué·e ? Dépliez l'indice. En dernier recours : `correction_exercice6.ipynb`.

## Énoncé

Créez deux tables **liées** : `Author` (`id`, `name`) et `Novel` (`id`, `title`, et une référence vers son auteur). Un auteur peut avoir plusieurs romans. Vous devez pouvoir écrire :

```python
auteur = Author(name="Isaac Asimov")
roman = Novel(title="Fondation", author=auteur)
```

… et retrouver `roman.author.name == "Isaac Asimov"` après un aller-retour dans une vraie base SQLite en mémoire (le test le vérifie).

In [ ]:
from sqlmodel import SQLModel, Field, Relationship
from sqlalchemy.orm import clear_mappers

## Votre code

In [ ]:
# Ne pas toucher : permet de relancer cette cellule sans erreur.
SQLModel.metadata.clear()
clear_mappers()

# TODO — Exercice 6 : créez Author (id, name, novels) et Novel (id, title,
#                     author_id en clé étrangère vers author.id, author).



<details>
<summary>💡 Indice</summary>

Il vous faut deux choses sur `Novel` : une **clé étrangère** (cherchez le paramètre `foreign_key` de `Field`, la table d'un auteur s'appelle par défaut `author`) et une **relation ORM** (cherchez `Relationship` et son paramètre `back_populates`). Pensez à la déclarer aussi côté `Author`, dans l'autre sens.
</details>

## Vérification

In [ ]:
import pytest

def get_fields(model):
    """Compatible pydantic v1 (SQLModel ancien) et v2 (SQLModel récent)."""
    if hasattr(model, "model_fields"):
        return model.model_fields
    return model.__fields__


def is_table(model) -> bool:
    return getattr(model, "__table__", None) is not None


# ---------------------------------------------------------------------------
# Exercice 6 — relation Author <-> Novel, testée avec une vraie base SQLite
# ---------------------------------------------------------------------------

def test_exercice6_classes_existent():
    assert hasattr(ex, "Author"), "La classe Author est introuvable."
    assert hasattr(ex, "Novel"), "La classe Novel est introuvable."


def test_exercice6_novel_a_la_cle_etrangere():
    fields = get_fields(ex.Novel)
    assert "author_id" in fields, (
        "Le champ 'author_id' est manquant sur Novel (clé étrangère vers author.id)."
    )


def test_exercice6_relation_fonctionne_reellement():
    """Crée une vraie base SQLite en mémoire, insère un auteur et un roman,
    puis vérifie que la relation ORM les relie correctement."""
    from sqlmodel import create_engine, Session

    engine = create_engine("sqlite://")
    SQLModel.metadata.create_all(engine)

    with Session(engine) as session:
        auteur = ex.Author(name="Isaac Asimov")
        roman = ex.Novel(title="Fondation", author=auteur)
        session.add(auteur)
        session.add(roman)
        session.commit()
        session.refresh(roman)

        assert roman.author is not None, (
            "roman.author est vide : vérifiez le Relationship() sur Novel."
        )
        assert roman.author.name == "Isaac Asimov", (
            "La relation ne pointe pas vers le bon auteur."
        )
        assert roman in auteur.novels, (
            "auteur.novels ne contient pas le roman : vérifiez back_populates des deux côtés."
        )

# --- Lancement des tests ----------------------------------------------------
import types

ex = types.SimpleNamespace(**globals())
_ok = 0
_tests = [(nom, f) for nom, f in list(globals().items()) if nom.startswith("test_")]
for _nom, _test in _tests:
    try:
        _test()
        print("✅", _nom)
        _ok += 1
    except Exception as erreur:
        print("❌", _nom, "→", str(erreur) or type(erreur).__name__)
print(f"\n{_ok}/{len(_tests)} tests réussis" + (" — bravo !" if _ok == len(_tests) else ""))
